# recortes_negativo_r2_fechamento_tese: a mistura (2 px + fechamento) com as perdas da tese

**Objetivo:** a mistura `recortes_negativo_r2_fechamento` (faixa do negativo de 2 px e fechamento de 3 px do marcador no treino) com as **perdas do
marcador da tese**, sem o TV: **Size** (penaliza a massa do marcador, empurra para marcadores pequenos) e **DMap** (penaliza
marcador perto da borda dos núcleos e no fundo, empurra para o centro). Na tese, essas perdas definem o que é um marcador bom para
um humano editar.

**Por que de novo:** sem o fechamento, o DMap (0,1) deixou o marcador menor, mas **picotado** (80% dos núcleos com o marcador
partido, 8% sem marcador) e baixou o Dice (`recortes_negativo_dmap`, 0,830). Na tese, quem segura os componentes é o L_comp, que não
foi implementado. Aqui o **fechamento** faz parte desse papel: as perdas agem sobre o marcador **cru** da MarkerUNet, e o fechamento
emenda os pedaços antes do ScribblePrompt. Pesos: Size 0,1 (λ_size/λ_seg da tese) e DMap 0,1 (o único peso já testado com recortes;
com 1/3000, o peso da tese, ele quase não age).

A comparação é com a mistura `recortes_negativo_r2_fechamento` (mesmos *folds*, sementes, faixa e fechamento; só as perdas mudam): o que importa é a
**editabilidade** (seção 8b) com o Dice sem queda relevante.

Configuração:

| Item | Valor |
|---|---|
| Perdas | soft Dice (1,0) + **Size (0,1)** + **DMap (0,1)** sobre o marcador cru da MarkerUNet; sem TV e sem Border |
| Fechamento | dilatação seguida de erosão do marcador por um disco de 3 px (`MarkerClosingStep`), antes do ScribblePrompt, no treino e na avaliação |
| Rede final | ScribblePrompt-UNet congelado, modo `positive` (τ = 0,5); entrada **128²** (o recorte de 256 px reduzido); **negativo = fundo do Cellpose** com faixa livre de 2 px (`cellpose_negative`); avaliação em blocos de 256 px com sobreposição de 1/4 e janela de Hann |
| MarkerUNet | U-Net (smp), encoder ResNet34 ImageNet, entrada RGBA (alpha = máscara do Cellpose) |
| Treino | **4 recortes de 256 × 256 px por imagem por época** (posição sorteada, depois da rotação e dos espelhamentos), lote de 16 recortes, MarkerUNet sem redução; Adam lr 1e-4, *grad clip* 1,0, *cosine schedule* |
| Protocolo | validação cruzada em 5 *folds* nas 37 imagens de treino (até 200 épocas, parada após 20 validações sem melhora do Dice); o modelo da melhor época de cada *fold* é gravado. **O teste não é usado aqui:** a configuração escolhida é avaliada uma única vez em `experimento_final.ipynb` |
| Linha de base | Cellpose sozinho, nas mesmas imagens |

**Saída:** `docs/estudo/resultados/recortes_negativo_r2/` (configuração e procedência, métricas por imagem da validação, curvas, resumo com a
comparação pareada com o Cellpose e figura com imagens da validação) e os modelos dos *folds* em
`MyDrive/cell-fuzzy-models/recortes_negativo_r2/` (no Colab). No Colab, a última célula baixa a pasta de resultados compactada.

Use um ambiente com **GPU**. Na GPU, o resultado pode variar um pouco entre execuções (operações não determinísticas do cuDNN).

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Configuração

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from src.data.load.preprocessed_dataset import NEGATIVE_KEY, load_preprocessed
from src.evaluation.figures import pick_cases, prediction_figure
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import save_run, summarize
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm, DMapTerm, SizeTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_closing_step import MarkerClosingStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.pipeline.tiled_pipeline import TiledPipeline
from src.training.checkpoints import fold_name, load_pipeline
from src.training.kfold import TrainConfig, final_epochs, run_kfold

EXPERIMENT_NAME = "recortes_negativo_r2_fechamento_tese"
OVERWRITE = False            # recusa gravar por cima de resultados ou modelos que já existem

RGBA_ALPHA = "mask"          # 4º canal da MarkerUNet: máscara do Cellpose ("prob": probabilidade)
TILE_SIZE = 256             # recorte do treino e bloco da avaliação (px da imagem original)
CROPS_PER_IMAGE = 4          # recortes sorteados de cada imagem por época
TILE_OVERLAP = 0.25          # sobreposição dos blocos na avaliação (fração do lado)
SP_INPUT_SIZE = (128, 128)   # entrada do ScribblePrompt (o recorte reduzido)
TAU = 0.5                    # limiar do canal positivo
RAIO_FECHAMENTO = 3          # fechamento do marcador antes do ScribblePrompt, em px (0: sem fechamento)
NEGATIVE_RADIUS = 2              # None: sem negativo; n: negativo = pixels a mais de n px de qualquer máscara do Cellpose

CONFIG = TrainConfig(
    k=5, batch_size=16, max_epochs=200, patience=20, min_delta=0.0,
    lr=1e-4, grad_clip=1.0, threshold=0.5, seed=42, log_every=10,
    crop_size=TILE_SIZE, crops_per_image=CROPS_PER_IMAGE,
)
DESCRIPTION = {
    "perdas": "soft Dice (1,0) + Size (0,1) + DMap (0,1) sobre o marcador cru, mapa de distância por núcleo; sem TV",
    "rede_final": f"ScribblePrompt-UNet v1 congelado, scribble_mode=positive, tau={TAU}, input_size={SP_INPUT_SIZE}"
                  + ("" if NEGATIVE_RADIUS is None
                     else f", negativo = fundo do Cellpose (faixa livre de {NEGATIVE_RADIUS} px)"),
    "fechamento": f"fechamento em tons de cinza do marcador por um disco de {RAIO_FECHAMENTO} px "
                  "antes do ScribblePrompt, no treino e na avaliação",
    "marker_net": "MarkerUNet (smp Unet, resnet34 ImageNet, 4 canais)",
    "rgba_alpha": RGBA_ALPHA,
    "aumentacao": "rot90 + flips, sorteada a cada época",
    "recortes": f"treino com {CROPS_PER_IMAGE} recortes de {TILE_SIZE}² por imagem por época (MarkerUNet sem redução); avaliação da imagem inteira em blocos de {TILE_SIZE}² com sobreposição {TILE_OVERLAP} e janela de Hann",
}

assert torch.cuda.is_available() or "google.colab" not in sys.modules, "No Colab, use um ambiente com GPU."
print("Device:", CONFIG.device)
print(json.dumps(DESCRIPTION, indent=2, ensure_ascii=False))

## 2. Onde os modelos são gravados

In [ ]:
# Modelos treinados: no Colab, no Google Drive de quem roda o notebook (sobrevive ao fim da sessão); fora dele,
# em ~/cell-fuzzy-models. Troque MODELS_ROOT para gravar em outro lugar.
if "google.colab" in sys.modules:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        MODELS_ROOT = "/content/drive/MyDrive/cell-fuzzy-models"
    except Exception as erro:
        print(f"Drive não montado ({erro}): os modelos ficam no disco da sessão e somem quando ela terminar.")
        MODELS_ROOT = "/content/cell-fuzzy-models"
else:
    MODELS_ROOT = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))
MODELS_DIR = os.path.join(MODELS_ROOT, EXPERIMENT_NAME)
print("Modelos em:", MODELS_DIR)

## 3. Dados de treino e linha de base do Cellpose

In [ ]:
train_samples = load_preprocessed("train", alpha=RGBA_ALPHA, negative_radius=NEGATIVE_RADIUS)
TRAIN_IDS = sorted(train_samples)
print(f"Treino: {len(train_samples)} imagens (o teste não é carregado neste notebook)")

d = [binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])["dice"] for s in train_samples.values()]
print(f"Cellpose sozinho, treino: Dice {np.mean(d):.4f} ± {np.std(d, ddof=1):.4f}")

## 4. Rede final (compartilhada) e montagem de cada experimento

In [ ]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_ROOT, "scribbleprompt"))
FINAL_NET = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=CONFIG.device, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)


def factory():
    """Monta um experimento novo: MarkerUNet recém-inicializada, pipeline em blocos e perdas.

    O treino passa recortes de TILE_SIZE² (a MarkerUNet os vê sem redução); a avaliação passa a imagem
    inteira, que o TiledPipeline divide em blocos de TILE_SIZE² e cola de volta. O ScribblePrompt é
    congelado e não guarda estado, então a mesma instância serve a todos os folds. Entre a MarkerUNet e o
    ScribblePrompt, o marcador passa pelo fechamento de RAIO_FECHAMENTO px (o gradiente atravessa o máximo e o
    mínimo); a chave markers continua sendo o marcador cru.
    """
    marker_net = MarkerUNet(config=UNET_CONFIG)
    pipeline = TrainingPipeline([
        MarkerStep(model=marker_net, differentiable=True, target_size=TILE_SIZE, device=CONFIG.device),
        MarkerClosingStep(radius=RAIO_FECHAMENTO),
        FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device, markers_key="markers_closed",
                               negative_key=None if NEGATIVE_RADIUS is None else NEGATIVE_KEY),
    ])
    composer = LossComposer([DiceTerm(), SizeTerm(weight=0.1), DMapTerm(weight=0.1)])
    return marker_net, TiledPipeline(pipeline, tile_size=TILE_SIZE, overlap=TILE_OVERLAP), composer

## 5. Validação cruzada

In [ ]:
folds = run_kfold(factory, train_samples, TRAIN_IDS, CONFIG, models_dir=MODELS_DIR, overwrite=OVERWRITE)

for f in folds:
    cp = np.mean([r["cellpose_dice"] for r in f.rows])
    print(f"fold {f.fold}: melhor época {f.best_epoch:3d} de {f.epochs_run:3d} | Dice val {f.best_dice:.4f} | Cellpose {cp:.4f}")

## 6. Registro e resumo

In [ ]:
N_EPOCHS = final_epochs(folds)
run_dir = save_run(EXPERIMENT_NAME, CONFIG, folds, final_epochs=N_EPOCHS, description=DESCRIPTION, overwrite=OVERWRITE)
print("Resultados em:", run_dir)
print("Épocas do treino final, se esta configuração for a escolhida (mediana das melhores épocas):", N_EPOCHS)

linhas = [r for f in folds for r in f.rows]
d = summarize(linhas)["dice"]
print(f"validação ({CONFIG.k} folds, {len(linhas)} imagens): Dice {d['mean']:.4f} ± {d['std']:.4f} "
      f"(IC95 {d['ci95'][0]:.4f}–{d['ci95'][1]:.4f}) | Cellpose {d['baseline_mean']:.4f} | "
      f"Δ {d['diff_mean']:+.4f} (IC95 {d['diff_ci95'][0]:+.4f}–{d['diff_ci95'][1]:+.4f}) | "
      f"melhora/piora {d['n_better']}/{d['n_worse']} | Wilcoxon p {d['wilcoxon_p']:.3g}")
print(f"marcador: média {np.mean([r['marker_mean_nucleo'] for r in linhas]):.3f} dentro dos núcleos do GT e "
      f"{np.mean([r['marker_mean_fundo'] for r in linhas]):.3f} fora")

### Como ler o resumo acima

- **`Resultados em`:** a pasta onde ficaram as métricas por imagem, as curvas, a configuração e a figura.
- **`Épocas do treino final`:** a mediana das melhores épocas dos 5 *folds*. Só é usada se esta configuração for a escolhida: o
  `experimento_final.ipynb` treina por esse número de épocas.
- **Linha `validação`** — cada uma das 37 imagens de treino foi prevista pelo modelo do *fold* em que ela ficou de fora, então
  nenhuma nota vem de uma imagem vista no treino:
  - **`Dice média ± desvio`:** o Dice médio das 37 imagens e o desvio padrão entre elas (quanto o resultado varia de imagem para
    imagem). Dice vai de 0 (nada em comum com o ground truth) a 1 (igual ao ground truth).
  - **`IC95`:** intervalo de confiança de 95% da média, por *bootstrap* (reamostragem das imagens): a faixa em que a média
    provavelmente estaria com outras imagens parecidas.
  - **`Cellpose`:** o Dice médio do Cellpose sozinho nas mesmas 37 imagens (a linha de base que o pipeline precisa superar).
  - **`Δ`:** a diferença média, imagem por imagem, entre o pipeline e o Cellpose (negativa = o pipeline perde), com o IC95 dela.
    Se o intervalo não cruza o zero, a diferença é consistente.
  - **`melhora/piora`:** em quantas imagens o pipeline ganhou e em quantas perdeu do Cellpose.
  - **`Wilcoxon p`:** teste pareado (as mesmas imagens nos dois métodos). Abaixo de 0,05, a diferença dificilmente é acaso.
- **Linha `marcador`:** o valor médio do marcador da MarkerUNet **dentro** dos núcleos do ground truth e **fora** deles. Se os dois
  forem parecidos, o marcador não distingue núcleo de fundo (por exemplo, um campo quase uniforme); quanto maior a diferença, mais
  o marcador "aponta" para os núcleos.

## 7. Imagens da validação: marcadores e segmentação

Três imagens da validação (a de pior, a mediana e a de melhor diferença de Dice em relação ao Cellpose), cada uma prevista pelo
modelo do *fold* em que ficou de fora, carregado da pasta dos modelos. Painéis: imagem, ground truth, máscara do Cellpose,
marcador (na escala 0–1 e com o contraste esticado), canal positivo que chega ao ScribblePrompt e segmentação. Recorte central de
400 px. A figura é salva na pasta do experimento.

In [ ]:
casos = [(rotulo, r, load_pipeline(factory, MODELS_DIR, fold_name(r["fold"]))[0])
         for rotulo, r in pick_cases(linhas)]
fig = prediction_figure(casos, train_samples, crop=400, threshold=CONFIG.threshold, device=CONFIG.device)
fig.savefig(os.path.join(run_dir, "figura_validacao.png"), dpi=80, bbox_inches="tight")
plt.show()

### Como ler a figura acima

Cada **linha** é uma imagem: imagens da **validação**, cada uma prevista pelo modelo do *fold* em que ela ficou de fora (carregado do Drive). As três foram escolhidas pela diferença de Dice contra o Cellpose: a **pior** (onde o
pipeline mais perde), a **mediana** e a **melhor** (onde mais ganha ou menos perde). O rótulo e o nome da imagem estão à esquerda.
Todos os painéis mostram o mesmo recorte central de 400 × 400 px da imagem de 1000 × 1000; os Dice dos títulos são da imagem
inteira.

| Coluna | O que mostra | Como ler |
|---|---|---|
| 1. imagem | a imagem H&E original | os núcleos são as manchas roxas escuras |
| 2. ground truth | a anotação manual: branco = núcleo, preto = fundo | é a resposta certa |
| 3. Cellpose | a segmentação do Cellpose sozinho (branco = núcleo) e o Dice dela | é a linha de base a superar |
| 4. marcador, 0–1 | a saída da MarkerUNet, na escala fixa de 0 (escuro) a 1 (claro); o título dá a fração da imagem com marcador ≥ 0,5 | mostra o nível do marcador; se tudo parecer da mesma cor, ele está quase constante (por exemplo, tudo perto de 0,5) |
| 5. marcador, esticado | o mesmo marcador com as cores esticadas entre os percentis 1 e 99 do recorte (o intervalo está no título) | revela a forma do marcador mesmo quando os valores são parecidos; um bom marcador fica claro sobre os núcleos da coluna 2 e escuro no fundo |
| 6. canal positivo, esticado | o que de fato entra no ScribblePrompt como "aqui há núcleo": `relu(m − 0,5)/0,5`, zero onde o marcador é menor que 0,5; cores esticadas de 0 ao máximo (no título) | um máximo baixo no título significa um positivo fraco; as partes pretas não indicam nada à rede |
| 7. segmentação | a saída final do ScribblePrompt, binarizada no limiar 0,5 (branco = núcleo), e o Dice dela | compare com a coluna 2 (o certo) e com a 3 (o Cellpose): núcleos inchados ou colados indicam excesso; núcleos faltando, falta |

O canal negativo, quando o experimento usa o do Cellpose, não aparece: ele é fixo (1 nos pixels a mais de alguns pixels de
qualquer máscara do Cellpose) e não depende do treino.

**Recortes:** a imagem inteira foi segmentada em blocos de 256² que se sobrepõem e foram colados com uma janela de Hann. Uma
textura em grade fraca no marcador (colunas 4 a 6) é o rastro dessa colagem; na segmentação ela não deveria aparecer.

## 8. Curvas e comparação por imagem

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for f in folds:
    axes[0].plot(f.train_loss_history, label=f"fold {f.fold}")
    axes[1].plot([h["epoch"] for h in f.val_dice_history], [h["dice"] for h in f.val_dice_history], label=f"fold {f.fold}")
axes[0].set_title("perda de treino")
axes[1].set_title("Dice de validação")
for ax in axes[:2]:
    ax.set_xlabel("época")
    ax.legend()

dice = np.array([r["dice"] for r in linhas])
base = np.array([r["cellpose_dice"] for r in linhas])
axes[2].scatter(base, dice)
lo, hi = min(dice.min(), base.min()) - 0.02, max(dice.max(), base.max()) + 0.02
axes[2].plot([lo, hi], [lo, hi], "k--", lw=1)
axes[2].set_xlabel("Dice do Cellpose")
axes[2].set_ylabel("Dice do pipeline")
axes[2].set_title("validação: uma imagem por ponto (acima da diagonal = melhor que o Cellpose)")
plt.tight_layout()
plt.show()

### Como ler os gráficos acima

- **Perda de treino** (esquerda): uma curva por *fold*, época a época. Ela deve cair e se estabilizar; o fim de cada curva é a
  época em que o treino daquele *fold* parou (20 validações seguidas sem melhora do Dice).
- **Dice de validação** (centro): o Dice médio das imagens que ficaram de fora em cada *fold*, época a época. O modelo guardado de
  cada *fold* é o da época do pico dessa curva, não o da última época.
- **Uma imagem por ponto** (direita): no eixo horizontal, o Dice do Cellpose; no vertical, o do pipeline, na mesma imagem. Pontos
  **acima da diagonal** tracejada são imagens em que o pipeline ganhou do Cellpose; **abaixo**, imagens em que perdeu. Quanto mais
  longe da diagonal, maior a diferença.

## 8b. Avaliação com a melhor inferência e editabilidade do marcador

Os modelos gravados dos *folds* (cada imagem prevista pelo modelo do *fold* em que ficou de fora), com o fechamento do treino, em
duas condições:

- **registro:** a avaliação da validação cruzada acima (sobreposição 1/4, sem TTA); tem de dar o mesmo Dice;
- **melhor inferência:** sobreposição de 1/2 e **TTA** (média das 8 rotações e espelhamentos).

Cada condição é comparada com o Cellpose, imagem por imagem. A editabilidade (`src/evaluation/editability.py`) é medida no
marcador cru da MarkerUNet (raio 0) e no marcador fechado que o ScribblePrompt recebe. As tabelas são gravadas em
`{run_dir}/melhor_inferencia.csv` e `{run_dir}/editabilidade.csv`. Na GPU, ~5 min.

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

from src.data.load.preprocessed_dataset import PreprocessedDataset, collate_samples
from src.evaluation.editability import marker_editability
from src.pipeline.steps.inference.marker_closing_step import grey_closing

SOBREPOSICAO_INFERENCIA = 0.5
IMAGENS_20X = {"TCGA-HE-7128-01Z-00-DX1", "TCGA-HE-7129-01Z-00-DX1", "TCGA-HE-7130-01Z-00-DX1"}
FOLDS = [(k, *load_pipeline(factory, MODELS_DIR, fold_name(k))) for k in range(CONFIG.k)]


def com_fechamento(tiled, raio, overlap):
    """O pipeline do fold com o fechamento de raio `raio` antes do ScribblePrompt (raio 0: sem fechamento)."""
    passos = [tiled.pipeline.steps[0]]
    if raio > 0:
        passos.append(MarkerClosingStep(radius=raio))
    passos.append(FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device, negative_key=NEGATIVE_KEY,
                                         markers_key="markers_closed" if raio > 0 else "markers"))
    return TiledPipeline(TrainingPipeline(passos), tile_size=tiled.tile_size, overlap=overlap)


def rodar(tiled, batch, tta):
    """Probabilidade (H, W) e marcador cru (H, W); com TTA, a probabilidade é a média das 8 rotações e espelhamentos."""
    with torch.no_grad():
        saida = tiled.run(batch, verbose=False)
        prob, marcador = saida["segmentation"], saida["markers"]
        if tta:
            for k in range(4):
                for espelhar in (False, True):
                    if k == 0 and not espelhar:
                        continue
                    def ida(x):
                        if not (torch.is_tensor(x) and x.ndim == 4):
                            return x
                        x = torch.rot90(x, k, dims=(-2, -1))
                        return torch.flip(x, dims=(-1,)) if espelhar else x
                    s = tiled.run({c: ida(v) for c, v in batch.items()}, verbose=False)["segmentation"]
                    if espelhar:
                        s = torch.flip(s, dims=(-1,))
                    prob = prob + torch.rot90(s, -k, dims=(-2, -1))
            prob = prob / 8
    return prob[0, 0].float().cpu().numpy(), marcador[0, 0].float().cpu().numpy()


CONDICOES = [("registro", RAIO_FECHAMENTO, TILE_OVERLAP, False),
             ("melhor inferência", RAIO_FECHAMENTO, SOBREPOSICAO_INFERENCIA, True)]
LINHAS, MARCADORES = [], {}
for condicao, raio, overlap, tta in CONDICOES:
    for k, tiled, meta in FOLDS:
        tiled = com_fechamento(tiled, raio, overlap)
        for image_id in meta["val_ids"]:
            batch = collate_samples([PreprocessedDataset(train_samples, ids=[image_id])[0]])
            batch = {c: (v.to(CONFIG.device) if torch.is_tensor(v) else v) for c, v in batch.items()}
            prob, marcador = rodar(tiled, batch, tta)
            if condicao == "melhor inferência":
                MARCADORES[image_id] = marcador
            s = train_samples[image_id]
            base = binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])
            LINHAS.append({"condicao": condicao, "id": image_id, "fold": k, "aumento_20x": image_id in IMAGENS_20X,
                           **binary_metrics(prob, s["ground_truth"]), **{f"cellpose_{m}": v for m, v in base.items()}})
melhor = pd.DataFrame(LINHAS)
melhor.to_csv(os.path.join(run_dir, "melhor_inferencia.csv"), index=False)

tabela = []
for condicao in dict.fromkeys(melhor.condicao):
    s = melhor[melhor.condicao == condicao]
    d = s.dice - s.cellpose_dice
    tabela.append({"condição": condicao, "Dice": s.dice.mean(), "precisão": s.precision.mean(),
                   "revocação": s.recall.mean(), "massa": s.mass_ratio.mean(), "Δ Cellpose": d.mean(),
                   "acima/abaixo do Cellpose": f"{int((d > 0).sum())}/{int((d < 0).sum())}",
                   "Wilcoxon p": wilcoxon(s.dice, s.cellpose_dice).pvalue, "Dice 20×": s[s.aumento_20x].dice.mean()})
print(f"Cellpose: Dice {melhor[melhor.condicao == 'registro'].cellpose_dice.mean():.4f}")
pd.set_option("display.width", 220)
pd.DataFrame(tabela).round(4)

In [ ]:
EDIT = []
for image_id, marcador in MARCADORES.items():
    for r in (0, RAIO_FECHAMENTO):
        fechado = marcador if r == 0 else grey_closing(torch.from_numpy(marcador), r).numpy()
        EDIT.append({"raio": r, "id": image_id, "area_marcador": float((fechado > TAU).mean()),
                     **marker_editability(fechado, train_samples[image_id]["ground_truth"], tau=TAU)})
edit = pd.DataFrame(EDIT)
edit.to_csv(os.path.join(run_dir, "editabilidade.csv"), index=False)
edit.drop(columns=["id"]).groupby("raio").mean().round(3)

### Como ler as tabelas acima

- **Primeira tabela:** uma linha por condição. **registro** tem de repetir o Dice da validação cruzada. **Δ Cellpose,
  acima/abaixo e Wilcoxon p** comparam com o Cellpose imagem por imagem. É aqui que se vê se a configuração passa do Cellpose.
- **Segunda tabela:** a editabilidade do marcador cru da MarkerUNet (raio 0) e do marcador fechado (3 px), o que o
  ScribblePrompt recebe. Mais regiões com um
  componente e menos fragmentos, buracos e uniões = marcador mais fácil de editar (ver `editabilidade_experimentos.ipynb`).

Para comparar: a mistura `recortes_negativo_r2_fechamento` (só Dice) está em `docs/estudo/resultados/recortes_negativo_r2_fechamento/`. O `recortes_negativo` (faixa 3 px, sem fechamento no treino) deu 0,8350 no registro; sem retreinar, com faixa de
2 px, sobreposição 1/2 e TTA, deu 0,8423 e, com o fechamento de 3 px, 0,8416, com 90% dos núcleos com um componente. Cellpose
0,8450.

## 9. Baixar os resultados (Colab)

In [ ]:
if "google.colab" in sys.modules:
    import shutil

    from google.colab import files

    archive = shutil.make_archive(f"/content/{EXPERIMENT_NAME}", "zip", root_dir=os.path.dirname(run_dir),
                                  base_dir=EXPERIMENT_NAME)
    files.download(archive)  # descompactar em docs/estudo/resultados/